# Flowracle: Out-of-Sample Evaluation on Held-Out Test Years (2021–2025)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/evaluate_flowracle.ipynb)

This standalone notebook loads a trained **Flowracle** ensemble checkpoint
(`xgb_1634_v135_ensemble.pkl`, produced by
[`train_flowracle.ipynb`](./train_flowracle.ipynb) using data strictly prior to
2021) together with the preprocessed Zarr datasets in `FLOWRACLE_DATA_DIR`, and
executes target-free out-of-sample inference and operational benchmarking across
California's snow-fed basins over the held-out `2021–2025` evaluation period.

### Overview
- **Strict Train / Test Separation**: No model fitting, hyperparameter tuning,
  climatology table estimation, or out-of-fold ensemble weight calculation
  occurs in this notebook. Every parameter is read directly from the frozen
  pre-2021 checkpoint, and ground-truth `target_fnf` arrays are popped from all
  evaluation batches before `predict(...)` is invoked.
- **Probabilistic Fan-Chart Hydrographs (`2021–2025`)**: Evaluates continuous
  monthly Full Natural Flow trajectories and 50% / 90% credible intervals across
  a top-down Sierra Nevada elevation transect (`WWR` at 2,697 m, `TLG` at
  1,789 m, and `AMF` at 1,320 m) across 1- to 6-month lead horizons.
- **Operational Bulletin 120 Benchmark (`2021–2025`)**: Compares April–July
  seasonal runoff volume forecasts issued on February 1, March 1, and April 1
  against official California Department of Water Resources (DWR) Bulletin 120
  forecasts across all 23 shared snow-fed basins via pinball quantile loss per
  issuance month, head-to-head quantile-loss win rates, cross-basin volume bias,
  and discretized Probability Integral Transform (PIT) calibration (`N = 225`
  paired evaluations across the 15 full-quantile basins).
- **Year-Round Monthly Pinball Loss & Hydrograph Timing (`WY 2021–2025`)**:
  Evaluates multi-quantile pinball loss broken down by calendar issuance month
  ($T \in \{1, \dots, 12\}$) and lead horizon ($l \in \{1, \dots, 6\}$),
  alongside water-year Center of Timing (`CT`), Spread of Timing (`SDoT`), and
  July–September runoff fraction (`f_summer`).

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

## 1. Environment Setup & Checkpoint Loading

`FLOWRACLE_DATA_DIR` specifies the directory populated by
`notebooks/data_preprocessing/`, and `FLOWRACLE_OUTPUT_DIR` specifies the
directory containing `xgb_1634_v135_ensemble.pkl` (saved by
`train_flowracle.ipynb`).

In [ ]:
!if [ ! -d "../flowracle" ] && [ ! -d "google-research/flowracle" ]; then git clone -n --depth=1 --filter=tree:0 https://github.com/google-research/google-research && (cd google-research && git sparse-checkout set --no-cone flowracle && git checkout); fi
!if [ -f "../requirements.txt" ]; then pip install -q -r ../requirements.txt; else pip install -q -r google-research/flowracle/requirements.txt; fi
!pip install -q --no-deps git+https://github.com/neuralgcm/coordax.git
!pip install -q --no-deps git+https://github.com/neuralgcm/terrax.git
!pip install -q chex fiddle jax jax_datetime

In [ ]:
import os
import sys

# Run JAX/Terrax data loading on CPU.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import jax

jax.config.update("jax_platform_name", "cpu")

import matplotlib.dates as mdates
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binom
from scipy.stats import chisquare
import xarray as xr

# Ensure the `flowracle` package directory is importable locally or on Colab.
REPO_ROOT = (
    os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(os.getcwd(), "..", "flowracle"))
    else os.path.abspath(
        os.path.join(os.getcwd(), "google-research", "flowracle")
    )
)
if REPO_ROOT not in sys.path:
  sys.path.insert(0, REPO_ROOT)

from flowracle import (
    QUANTILES,
    build_terrax_dataloader,
    compute_hydrograph_timing_metrics,
    consolidate_targets,
    evaluate_against_b120,
    get_issuance_times,
    load_model,
    predict,
    quantile_loss,
    resolve_dataset_path,
)
from flowracle.data_pipeline import SYNTHETIC_MONTH_STEP

DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
MODEL_DIR = os.environ.get("FLOWRACLE_OUTPUT_DIR", "./model_output")
print("FLOWRACLE_DATA_DIR:", DATA_DIR)
print("FLOWRACLE_OUTPUT_DIR:", MODEL_DIR)

loaded_model = load_model(model_dir=MODEL_DIR)
print(
    "Loaded Flowracle checkpoint: "
    f"elev={len(loaded_model['models_elev'])} quantiles, "
    f"melt={len(loaded_model['models_melt'])} quantiles, "
    f"resid={len(loaded_model['models_resid'])} quantiles "
    f"(grid={QUANTILES[0]:.2f}..{QUANTILES[-1]:.2f}, "
    f"sigma2_resid={loaded_model.get('sigma2_resid', 0.0):.4f})"
)

## 2. Held-Out 2021–2025 Data Stream & Target-Free Inference

To evaluate all 60 monthly issuances from **January 1, 2021** through
**December 1, 2025**, the evaluation slice spans `2020-07-01` to `2026-05-01`:
- Prepending July–December 2020 (`2020-07-01`) populates the 6-month trailing
  predictor window (`lags [-5, ..., 0]`) for the January 1, 2021 issuance.
- Extending the target stencil upper bound to `2026-05-01` accommodates the
  6-month lead horizon (`leads [+1, ..., +6]`) of the December 1, 2025 issuance.
- Ground-truth `target_fnf` arrays are explicitly stripped from every batch
  prior to calling `predict(...)`, and predictions are aligned to the 23
  snow-fed watersheds covered by California Bulletin 120.

In [ ]:
aligned = build_terrax_dataloader(data_dir=DATA_DIR)

oos_end = min(aligned.real_time[-1], np.datetime64("2026-05-01"))
eval_list_oos = aligned.dataloader.build_eval_inputs(
    aligned.inputs_spec,
    time_sample_offset=SYNTHETIC_MONTH_STEP,
    dataset_time_slice=aligned.real_slice_to_synth(
        (np.datetime64("2020-07-01"), oos_end)
    ),
    batch_size_per_device=1,
)
oos_y_true = consolidate_targets(
    iter(eval_list_oos), source="target_fnf", field="fnf", axis=0
)
oos_times = get_issuance_times(iter(eval_list_oos))

# Strip ground-truth targets from all evaluation batches before model inference.
for batch_item in eval_list_oos:
  batch_item.pop("target_fnf", None)

oos_y_pred = predict(eval_list_oos, aux_ds=aligned.aux_ds, model=loaded_model)

# Align the 23 snow-fed watersheds shared with California Bulletin 120.
b120_path = resolve_dataset_path(
    "b120-forecasts", "b120_consolidated.zarr", DATA_DIR
)
b120_da = xr.open_zarr(b120_path)["forecast_fnf_april_july"]
model_station_ids = aligned.aux_ds.station_id.values.tolist()
b120_station_ids = b120_da.station_id.values.tolist()
shared_pairs = [
    (b_idx, model_station_ids.index(stn))
    for b_idx, stn in enumerate(b120_station_ids)
    if stn in model_station_ids
]
b120_keep_idx = [p[0] for p in shared_pairs]
b120_to_model_idx = [p[1] for p in shared_pairs]
b120_da_aligned = b120_da.isel(station_id=b120_keep_idx)

oos_y_pred_23 = oos_y_pred[..., b120_to_model_idx]
oos_y_true_23 = oos_y_true[..., b120_to_model_idx]
fnf_ds_23 = aligned.fnf_ds.sel(
    station_id=aligned.aux_ds.station_id.values[b120_to_model_idx]
)

print(
    f"Held-out issuances: {str(oos_times[0])[:10]} to {str(oos_times[-1])[:10]}"
    f" ({len(oos_times)} monthly issuances)"
)
print(f"Full 25-station forecast shape (M, T, L, S): {oos_y_pred.shape}")
print(f"23 snow-fed basin shape        (M, T, L, S): {oos_y_pred_23.shape}")

## 3. Probabilistic Fan-Chart Hydrographs Across the Elevation Transect (2021–2025)

Continuous probabilistic fan charts display the monthly Full Natural Flow (FNF)
trajectory over the `2021–2025` test period across a Sierra Nevada elevation
transect:
- **High Elevation**: West Walker River (`WWR`, 2,697 m — subalpine headwater)
- **Mid Elevation**: Tuolumne River (`TLG`, 1,789 m — mid-Sierra snowpack)
- **Low Elevation**: American River (`AMF`, 1,320 m — rain–snow transition)

Shaded envelopes denote the 90% (`q05–q95`) and 50% (`q25–q75`) credible
intervals, solid navy lines show the ensemble median (`q50`), and dashed black
markers denote observed monthly FNF across primary operational horizons (Leads 1
and 3 months) and extended horizons (Leads 2, 4, and 6 months).

In [ ]:
def plot_probabilistic_fan_charts(
    eval_times,
    y_pred,
    y_true,
    fnf_ds,
    leads_to_plot=None,
    max_target_date="2025-12-01",
):
  """Renders continuous probabilistic fan charts across the elevation transect."""
  if leads_to_plot is None:
    leads_to_plot = [(0, "Lead 1 Month"), (2, "Lead 3 Months")]

  eval_dates = pd.to_datetime(eval_times)
  station_names = list(fnf_ds.station_id.values)
  n_stations = y_pred.shape[-1]
  if len(station_names) > n_stations:
    station_names = station_names[:n_stations]

  transect = [
      ("WWR", "West Walker River (WWR, 2,697 m)"),
      ("TLG", "Tuolumne River (TLG, 1,789 m)"),
      ("AMF", "American River (AMF, 1,320 m)"),
  ]

  max_dt = pd.Timestamp(max_target_date)
  scale_y = 1e5  # 10^5 Acre-Feet
  q05 = np.percentile(y_pred, 5, axis=0) / scale_y
  q25 = np.percentile(y_pred, 25, axis=0) / scale_y
  q50 = np.percentile(y_pred, 50, axis=0) / scale_y
  q75 = np.percentile(y_pred, 75, axis=0) / scale_y
  q95 = np.percentile(y_pred, 95, axis=0) / scale_y
  y_true_scaled = y_true / scale_y

  n_cols = len(leads_to_plot)
  fig, axes = plt.subplots(
      nrows=3,
      ncols=n_cols,
      figsize=(6.6 * n_cols, 9.0),
      sharex=True,
      sharey="row",
  )
  if n_cols == 1:
    axes = axes[:, None]

  t_start = pd.Timestamp("2021-01-01")
  t_end = max_dt

  for row, (stn_code, stn_title) in enumerate(transect):
    stn_idx = (
        station_names.index(stn_code)
        if stn_code in station_names
        else min(row, n_stations - 1)
    )
    for col, (l_idx, l_label) in enumerate(leads_to_plot):
      ax = axes[row, col]
      target_dates = np.array(
          [d + pd.DateOffset(months=l_idx) for d in eval_dates]
      )
      keep = (target_dates >= t_start) & (target_dates <= t_end)
      td_plot = target_dates[keep]
      is_first = row == 0 and col == 0

      ax.fill_between(
          td_plot,
          q05[keep, l_idx, stn_idx],
          q95[keep, l_idx, stn_idx],
          color="skyblue",
          alpha=0.35,
          label="90% Credible Interval" if is_first else None,
      )
      ax.fill_between(
          td_plot,
          q25[keep, l_idx, stn_idx],
          q75[keep, l_idx, stn_idx],
          color="dodgerblue",
          alpha=0.45,
          label="50% Credible Interval" if is_first else None,
      )
      ax.plot(
          td_plot,
          q50[keep, l_idx, stn_idx],
          color="navy",
          lw=1.8,
          label="Flowracle Median" if is_first else None,
      )
      ax.plot(
          td_plot,
          y_true_scaled[keep, l_idx, stn_idx],
          color="black",
          linestyle="--",
          marker="o",
          ms=3.2,
          lw=1.3,
          label="Observed FNF" if is_first else None,
      )

      ax.set_title(f"{stn_title}: {l_label}", fontsize=11.5, fontweight="bold")
      ax.set_xlim([t_start, t_end])
      ax.xaxis.set_major_locator(mdates.YearLocator())
      ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
      ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
      ax.grid(True, linestyle=":", alpha=0.6)
      if col == 0:
        ax.set_ylabel("FNF ($10^5$ AF)", fontsize=11.5)
      if row == 2:
        ax.set_xlabel("Year", fontsize=11.5)

    cur_top = max(axes[row, c].get_ylim()[1] for c in range(n_cols))
    axes[row, 0].set_ylim(0.0, cur_top)

  handles, labels = axes[0, 0].get_legend_handles_labels()
  fig.legend(
      handles,
      labels,
      loc="lower center",
      bbox_to_anchor=(0.5, 0.005),
      ncol=4,
      fontsize=11,
      framealpha=0.95,
  )
  plt.tight_layout(rect=[0, 0.05, 1, 1.0])
  plt.show()
  del fig


# Primary horizons: Lead 1 Month & Lead 3 Months
plot_probabilistic_fan_charts(
    oos_times,
    oos_y_pred_23,
    oos_y_true_23,
    fnf_ds_23,
    leads_to_plot=[(0, "Lead 1 Month"), (2, "Lead 3 Months")],
)

# Extended horizons: Lead 2, Lead 4 & Lead 6 Months
plot_probabilistic_fan_charts(
    oos_times,
    oos_y_pred_23,
    oos_y_true_23,
    fnf_ds_23,
    leads_to_plot=[
        (1, "Lead 2 Months"),
        (3, "Lead 4 Months"),
        (5, "Lead 6 Months"),
    ],
)

## 4. Operational California DWR Bulletin 120 Benchmark (2021–2025)

Cumulative April–July (AMJJ) seasonal runoff predictions issued on February 1,
March 1, and April 1 (`2021–2025`, `N = 23` snow-fed watersheds, `115`
basin-years per issuance month, `345` total) are evaluated against official
Bulletin 120 forecasts:
- For the 15 watersheds with published Bulletin 120 prediction intervals, each
  trial is scored by average pinball quantile loss across
  $\tau \in \{0.10, 0.50, 0.90\}$; for the remaining 8 median-only
  watersheds, both systems are scored strictly on median pinball loss
  ($\tau = 0.50$).
- The evaluation below reports mean pinball loss ($\pm 1$ SEM across
  watersheds), relative error reduction, head-to-head quantile-loss win rates,
  cross-basin volume bias, and discretized Probability Integral Transform (PIT)
  calibration (`N = 225` paired evaluations across the 15 full-quantile
  basins).

In [ ]:
b120_summary = evaluate_against_b120(
    eval_list_oos,
    oos_y_pred,
    oos_y_true,
    aligned.aux_ds,
    b120_da=b120_da,
    plot=False,
)
print(
    "15-basin tri-quantile (q=0.1, 0.5, 0.9) mean pinball loss (Feb/Mar/Apr):"
    f" Flowracle={[round(v, 1) for v in b120_summary['ours_15basin_3q_losses']]}"
    " AF vs"
    f" B120={[round(v, 1) for v in b120_summary['b120_15basin_3q_losses']]}"
    " AF\n"
)

oos_dates = pd.DatetimeIndex(oos_times)
quantiles_b120 = (0.1, 0.5, 0.9)
month_labels = {2: "February 1", 3: "March 1", 4: "April 1"}

ours_ql_23, b120_ql_23 = [], []
ours_sem_23, b120_sem_23 = [], []
stn_biases_b120_by_m, stn_biases_ours_by_m = [], []
all_ours_stn_losses, all_b120_stn_losses = [], []
pit_targets, pit_ours_q10, pit_ours_q50, pit_ours_q90 = [], [], [], []
pit_b120_q10, pit_b120_q50, pit_b120_q90 = [], [], []

print("=" * 98)
print(
    "SPRING APRIL-JULY PINBALL QUANTILE LOSS BENCHMARK AGAINST BULLETIN 120"
    " (2021-2025, 23 BASINS)"
)
print("-" * 98)
print(
    f"{'Issuance':<12} | {'B120 Loss +/- SEM':>21} | {'Ours Loss +/- SEM':>21}"
    f" | {'Reduction':>9} | {'Head-to-Head Wins':>19}"
)
print("-" * 98)

for m in (2, 3, 4):
  m_mask = (oos_dates.month == m) & oos_dates.year.isin(range(2021, 2026))
  m_times = oos_dates[m_mask].values
  amjj_idx = np.arange(4 - m, 8 - m)

  ours_amjj = oos_y_pred_23[:, m_mask, amjj_idx[0] : amjj_idx[-1] + 1, :].sum(
      axis=2
  )
  true_amjj = oos_y_true_23[m_mask, amjj_idx[0] : amjj_idx[-1] + 1, :].sum(
      axis=1
  )

  valid_t = np.isin(m_times, b120_da_aligned.time.values) & (
      ~np.isnan(true_amjj).all(axis=1)
  )
  ours_amjj = ours_amjj[:, valid_t, :]
  true_amjj = true_amjj[valid_t, :]
  b120_m = b120_da_aligned.sel(time=m_times[valid_t])

  has_3q = ~b120_m.sel(quantile=0.1).isnull().all(dim="time").values
  q15_ids = np.where(has_3q)[0]
  pit_targets.append(true_amjj[:, q15_ids])
  pit_b120_q10.append(b120_m.sel(quantile=0.1).isel(station_id=q15_ids).values)
  pit_b120_q50.append(b120_m.sel(quantile=0.5).isel(station_id=q15_ids).values)
  pit_b120_q90.append(b120_m.sel(quantile=0.9).isel(station_id=q15_ids).values)
  pit_ours_q10.append(np.quantile(ours_amjj[:, :, q15_ids], 0.1, axis=0))
  pit_ours_q50.append(np.quantile(ours_amjj[:, :, q15_ids], 0.5, axis=0))
  pit_ours_q90.append(np.quantile(ours_amjj[:, :, q15_ids], 0.9, axis=0))

  n_t, n_s = true_amjj.shape
  ours_stn_loss = np.zeros((n_t, n_s))
  b120_stn_loss = np.zeros((n_t, n_s))
  o_med = np.quantile(ours_amjj, 0.5, axis=0)
  b_med = b120_m.sel(quantile=0.5).values

  for s_i in range(n_s):
    if has_3q[s_i]:
      for q in quantiles_b120:
        err_o = true_amjj[:, s_i] - np.quantile(ours_amjj[:, :, s_i], q, axis=0)
        err_b = true_amjj[:, s_i] - b120_m.sel(quantile=q).values[:, s_i]
        ours_stn_loss[:, s_i] += np.maximum(q * err_o, (q - 1.0) * err_o) / 3.0
        b120_stn_loss[:, s_i] += np.maximum(q * err_b, (q - 1.0) * err_b) / 3.0
    else:
      err_o = true_amjj[:, s_i] - o_med[:, s_i]
      err_b = true_amjj[:, s_i] - b_med[:, s_i]
      ours_stn_loss[:, s_i] = np.maximum(0.5 * err_o, -0.5 * err_o)
      b120_stn_loss[:, s_i] = np.maximum(0.5 * err_b, -0.5 * err_b)

  mean_o = float(np.mean(ours_stn_loss))
  mean_b = float(np.mean(b120_stn_loss))
  stn_means_o = np.mean(ours_stn_loss, axis=0)
  stn_means_b = np.mean(b120_stn_loss, axis=0)
  sem_o = float(np.std(stn_means_o, ddof=1) / np.sqrt(n_s))
  sem_b = float(np.std(stn_means_b, ddof=1) / np.sqrt(n_s))

  ours_ql_23.append(mean_o)
  b120_ql_23.append(mean_b)
  ours_sem_23.append(sem_o)
  b120_sem_23.append(sem_b)
  all_ours_stn_losses.append(ours_stn_loss)
  all_b120_stn_losses.append(b120_stn_loss)
  stn_biases_b120_by_m.append(np.nanmean(b_med - true_amjj, axis=0) / 1e4)
  stn_biases_ours_by_m.append(np.nanmean(o_med - true_amjj, axis=0) / 1e4)

  wins = int(np.sum(ours_stn_loss < b120_stn_loss))
  trials = int(ours_stn_loss.size)
  red_pct = (1.0 - mean_o / max(mean_b, 1e-8)) * 100.0
  print(
      f"{month_labels[m]:<12} |"
      f" {mean_b:>9,.0f} +/- {sem_b:>6,.0f} |"
      f" {mean_o:>9,.0f} +/- {sem_o:>6,.0f} |"
      f" {red_pct:>8.1f}% |"
      f" {wins:>3d}/{trials:<3d} ({100.0 * wins / trials:4.1f}%)"
  )

pooled_o = np.concatenate(all_ours_stn_losses, axis=0)
pooled_b = np.concatenate(all_b120_stn_losses, axis=0)
pooled_wins = int(np.sum(pooled_o < pooled_b))
pooled_trials = int(pooled_o.size)
pooled_red = (1.0 - float(np.mean(pooled_o)) / float(np.mean(pooled_b))) * 100.0
basins_gt_50 = int(np.sum(np.mean(pooled_o < pooled_b, axis=0) > 0.5))
print("-" * 98)
print(
    f"{'Pooled':<12} |"
    f" {float(np.mean(pooled_b)):>9,.0f} AF          |"
    f" {float(np.mean(pooled_o)):>9,.0f} AF          |"
    f" {pooled_red:>8.1f}% |"
    f" {pooled_wins:>3d}/{pooled_trials:<3d}"
    f" ({100.0 * pooled_wins / pooled_trials:4.1f}%)"
)
print(f"Watersheds with >50% head-to-head win rate: {basins_gt_50}/23")
print("=" * 98)


def _pit_counts(t_v, q10_v, q50_v, q90_v):
  return np.array([
      np.mean(t_v <= q10_v),
      np.mean((t_v > q10_v) & (t_v <= q50_v)),
      np.mean((t_v > q50_v) & (t_v <= q90_v)),
      np.mean(t_v > q90_v),
  ])


t_flat = np.concatenate(pit_targets, axis=0).ravel()
o10_flat = np.concatenate(pit_ours_q10, axis=0).ravel()
o50_flat = np.concatenate(pit_ours_q50, axis=0).ravel()
o90_flat = np.concatenate(pit_ours_q90, axis=0).ravel()
b10_flat = np.concatenate(pit_b120_q10, axis=0).ravel()
b50_flat = np.concatenate(pit_b120_q50, axis=0).ravel()
b90_flat = np.concatenate(pit_b120_q90, axis=0).ravel()
valid_pit = ~np.isnan(t_flat) & ~np.isnan(b10_flat)
t_v = t_flat[valid_pit]
o10_v, o50_v, o90_v = (
    o10_flat[valid_pit],
    o50_flat[valid_pit],
    o90_flat[valid_pit],
)
b10_v, b50_v, b90_v = (
    b10_flat[valid_pit],
    b50_flat[valid_pit],
    b90_flat[valid_pit],
)

ours_pit = _pit_counts(t_v, o10_v, o50_v, o90_v)
b120_pit = _pit_counts(t_v, b10_v, b50_v, b90_v)
ideal_pit = np.array([0.10, 0.40, 0.40, 0.10])
n_pit = len(t_v)

ideal_ci = np.zeros((2, 4))
for b_i, p_nom in enumerate(ideal_pit):
  ideal_ci[0, b_i] = max(0.0, p_nom - binom.ppf(0.025, n_pit, p_nom) / n_pit)
  ideal_ci[1, b_i] = max(0.0, binom.ppf(0.975, n_pit, p_nom) / n_pit - p_nom)

rng = np.random.RandomState(42)
boot_o = np.zeros((2000, 4))
boot_b = np.zeros((2000, 4))
for b_iter in range(2000):
  idx = rng.choice(n_pit, size=n_pit, replace=True)
  boot_o[b_iter] = _pit_counts(t_v[idx], o10_v[idx], o50_v[idx], o90_v[idx])
  boot_b[b_iter] = _pit_counts(t_v[idx], b10_v[idx], b50_v[idx], b90_v[idx])
ours_yerr = np.array([
    np.maximum(0.0, ours_pit - np.percentile(boot_o, 2.5, axis=0)),
    np.maximum(0.0, np.percentile(boot_o, 97.5, axis=0) - ours_pit),
])
b120_yerr = np.array([
    np.maximum(0.0, b120_pit - np.percentile(boot_b, 2.5, axis=0)),
    np.maximum(0.0, np.percentile(boot_b, 97.5, axis=0) - b120_pit),
])

chi_o, p_o = chisquare(ours_pit * n_pit, f_exp=ideal_pit * n_pit)
chi_b, p_b = chisquare(b120_pit * n_pit, f_exp=ideal_pit * n_pit)
print(
    f"\n15-Basin PIT Calibration (N = {n_pit}):"
    f" Flowracle={np.round(ours_pit, 3).tolist()} (Chi2={chi_o:.2f},"
    f" p={p_o:.3f}) | B120={np.round(b120_pit, 3).tolist()} (Chi2={chi_b:.2f},"
    f" p={p_b:.2e})"
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

# Panel (a): Discretized PIT histogram
x_bins = np.arange(4)
w_bar = 0.25
axes[0].bar(
    x_bins - w_bar,
    ideal_pit,
    w_bar,
    yerr=ideal_ci,
    capsize=3.5,
    color="#cccccc",
    edgecolor="#666666",
    label="Nominal (95% CI)",
)
axes[0].bar(
    x_bins,
    ours_pit,
    w_bar,
    yerr=ours_yerr,
    capsize=3.5,
    color="steelblue",
    edgecolor="#1c3d5a",
    label="Flowracle (95% CI)",
)
axes[0].bar(
    x_bins + w_bar,
    b120_pit,
    w_bar,
    yerr=b120_yerr,
    capsize=3.5,
    color="crimson",
    edgecolor="#67000d",
    label="Bulletin 120 (95% CI)",
)
axes[0].set_xticks(x_bins)
axes[0].set_xticklabels(["<= Q10", "Q10-Q50", "Q50-Q90", "> Q90"])
axes[0].set_ylim(0.0, 0.70)
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("PIT Interval")
axes[0].set_title("a) Probabilistic Calibration (15 Basins)")
axes[0].legend(fontsize=9, loc="upper right")
axes[0].grid(axis="y", alpha=0.3)

# Panel (b): Mean pinball quantile loss by issuance month (+/- 1 SEM)
x_m = np.arange(3, dtype=float)
axes[1].errorbar(
    x_m - 0.03,
    np.array(b120_ql_23) / 1e4,
    yerr=np.array(b120_sem_23) / 1e4,
    marker="o",
    linewidth=2.0,
    capsize=4,
    color="crimson",
    label="Bulletin 120 (Mean +/- SEM)",
)
axes[1].errorbar(
    x_m + 0.03,
    np.array(ours_ql_23) / 1e4,
    yerr=np.array(ours_sem_23) / 1e4,
    marker="s",
    linewidth=2.0,
    capsize=4,
    color="steelblue",
    label="Flowracle (Mean +/- SEM)",
)
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(["Feb", "Mar", "Apr"])
axes[1].set_xlabel("Issuance Month")
axes[1].set_ylabel("Mean Pinball Loss ($10^4$ AF)")
axes[1].set_title("b) Pinball Quantile Loss by Issuance Month (23 Basins)")
axes[1].legend(fontsize=9, loc="upper right")
axes[1].grid(axis="y", alpha=0.3)

# Panel (c): Cross-basin forecast volume bias boxplots
pos_b = x_m - 0.16
pos_o = x_m + 0.16
axes[2].boxplot(
    stn_biases_b120_by_m,
    positions=pos_b,
    widths=0.25,
    patch_artist=True,
    boxprops=dict(facecolor="#ff9999", color="crimson", alpha=0.85),
    medianprops=dict(color="#67000d", lw=2.0),
    whiskerprops=dict(color="crimson"),
    capprops=dict(color="crimson"),
)
axes[2].boxplot(
    stn_biases_ours_by_m,
    positions=pos_o,
    widths=0.25,
    patch_artist=True,
    boxprops=dict(facecolor="#a6c8e0", color="steelblue", alpha=0.85),
    medianprops=dict(color="#1c3d5a", lw=2.0),
    whiskerprops=dict(color="steelblue"),
    capprops=dict(color="steelblue"),
)
axes[2].axhline(0.0, color="gray", linestyle="--", lw=1.2)
axes[2].set_xticks(range(3))
axes[2].set_xticklabels(["Feb", "Mar", "Apr"])
axes[2].set_xlabel("Issuance Month")
axes[2].set_ylabel("Forecast Volume Bias ($10^4$ AF)")
axes[2].set_title("c) Cross-Basin Volume Bias (23 Basins)")
axes[2].legend(
    handles=[
        Patch(facecolor="#ff9999", edgecolor="crimson", label="Bulletin 120"),
        Patch(facecolor="#a6c8e0", edgecolor="steelblue", label="Flowracle"),
    ],
    fontsize=9,
    loc="upper left",
)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
del fig

## 5. Year-Round Monthly Pinball Loss by Issuance Month & Water-Year Hydrograph Timing (`WY 2021–2025`)

Beyond the spring April–July season, Flowracle issues monthly forecasts on the
1st of every calendar month ($T \in \{1, \dots, 12\}$) across lead horizons
$l \in \{1, \dots, 6\}$:
1. **Monthly Pinball Quantile Loss by Issuance Month (`Jan–Dec`, `2021–2025`)**:
   Evaluates mean pinball loss across all 13 predicted quantile levels
   ($\tau \in \{0.01, 0.05, 0.10, \dots, 0.99\}$) and median pinball loss
   ($\tau = 0.50$) across the 23 snow-fed basins for each calendar issuance
   month and lead horizon.
2. **Water-Year Hydrograph Timing (`WY 2021–2025`, Oct 2020 – Sep 2025)**:
   Reconstructs complete 12-month water-year hydrographs from rolling median
   forecasts at leads $l \in \{1, 2, 3\}$ (prepending August–December 2020
   spin-up issuances so October–December 2020 targets of `WY 2021` are covered
   across all leads) and compares Center of Timing (`CT`), Spread of Timing
   (`SDoT`), and July–September runoff fraction (`f_summer`) against observed
   `WY 2021–2025` streamflow.

In [ ]:
last_obs_month = min(
    pd.Timestamp(aligned.real_time[-7]), pd.Timestamp("2025-12-01")
)

# Pre-extract the 13 predicted quantile surfaces from the 100-member ensemble.
pred_quantiles_23 = np.quantile(
    oos_y_pred_23, QUANTILES, axis=0
)  # (13, T, 6, 23)
med_idx = int(np.where(np.isclose(QUANTILES, 0.5))[0][0])
cal_names = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
ql_by_month_lead = np.full((12, 6), np.nan)
med_ql_by_month = np.full(12, np.nan)

print("=" * 84)
print(
    "MONTHLY 13-QUANTILE PINBALL LOSS BY ISSUANCE MONTH ACROSS 23 BASINS "
    "(2021-2025, ACRE-FEET)"
)
print("-" * 84)
print(
    f"{'Issuance':<10} | {'Lead 1M':>11} | {'Lead 3M':>11} | {'Lead 6M':>11} |"
    f" {'6-Lead Mean':>12} | {'Median (q=0.5)':>14}"
)
print("-" * 84)

for cal_m in range(1, 13):
  iss_mask = oos_dates.month == cal_m
  iss_indices = np.where(iss_mask)[0]
  lead_losses = []
  med_losses = []
  for l_idx in range(6):
    valid_rows = [
        idx
        for idx in iss_indices
        if pd.Timestamp("2021-01-01")
        <= (pd.Timestamp(oos_times[idx]) + pd.DateOffset(months=l_idx))
        <= last_obs_month
    ]
    if not valid_rows:
      continue
    y_t = oos_y_true_23[valid_rows, l_idx, :]
    q_losses = [
        quantile_loss(
            pred_quantiles_23[q_i, valid_rows, l_idx, :], y_t, float(tau)
        )
        for q_i, tau in enumerate(QUANTILES)
    ]
    ql_by_month_lead[cal_m - 1, l_idx] = float(np.mean(q_losses))
    lead_losses.append(ql_by_month_lead[cal_m - 1, l_idx])
    med_losses.append(
        quantile_loss(
            pred_quantiles_23[med_idx, valid_rows, l_idx, :], y_t, 0.5
        )
    )
  med_ql_by_month[cal_m - 1] = float(np.mean(med_losses))
  print(
      f"{cal_names[cal_m - 1] + ' 1':<10} |"
      f" {ql_by_month_lead[cal_m - 1, 0]:>11,.1f} |"
      f" {ql_by_month_lead[cal_m - 1, 2]:>11,.1f} |"
      f" {ql_by_month_lead[cal_m - 1, 5]:>11,.1f} |"
      f" {float(np.nanmean(lead_losses)):>12,.1f} |"
      f" {med_ql_by_month[cal_m - 1]:>14,.1f}"
  )
print("=" * 84)

fig, ax = plt.subplots(figsize=(9.5, 4.5))
x_cal = np.arange(12)
ax.plot(
    x_cal,
    ql_by_month_lead[:, 0] / 1e4,
    marker="o",
    lw=2.0,
    color="navy",
    label="Lead 1 Month",
)
ax.plot(
    x_cal,
    ql_by_month_lead[:, 2] / 1e4,
    marker="s",
    lw=2.0,
    color="steelblue",
    label="Lead 3 Months",
)
ax.plot(
    x_cal,
    ql_by_month_lead[:, 5] / 1e4,
    marker="^",
    lw=2.0,
    color="darkorange",
    label="Lead 6 Months",
)
ax.plot(
    x_cal,
    np.nanmean(ql_by_month_lead, axis=1) / 1e4,
    linestyle="--",
    lw=1.8,
    color="black",
    label="6-Lead Mean",
)
ax.set_xticks(x_cal)
ax.set_xticklabels(cal_names)
ax.set_xlabel("Forecast Issuance Month")
ax.set_ylabel("Mean 13-Quantile Pinball Loss ($10^4$ AF)")
ax.set_title(
    "Monthly Pinball Quantile Loss by Issuance Month (2021-2025, 23 Basins)"
)
ax.legend(ncol=2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
del fig

# Predict only the Aug-Dec 2020 spin-up issuances needed for Oct-Dec 2020
# targets of WY 2021 at leads l in {1, 2, 3}, then combine with `oos_y_pred_23`.
wy2021_spinup_list = aligned.dataloader.build_eval_inputs(
    aligned.inputs_spec,
    time_sample_offset=SYNTHETIC_MONTH_STEP,
    dataset_time_slice=aligned.real_slice_to_synth(
        (np.datetime64("2020-01-01"), np.datetime64("2021-05-31"))
    ),
    batch_size_per_device=1,
)
spinup_times = get_issuance_times(iter(wy2021_spinup_list))
for batch_item in wy2021_spinup_list:
  batch_item.pop("target_fnf", None)
spinup_pred_med = np.median(
    predict(wy2021_spinup_list, aux_ds=aligned.aux_ds, model=loaded_model)[
        ..., b120_to_model_idx
    ],
    axis=0,
)
oos_pred_med_23 = np.median(oos_y_pred_23, axis=0)

pred_med_by_issuance = {
    pd.Timestamp(t): spinup_pred_med[i] for i, t in enumerate(spinup_times)
}
for i, t in enumerate(oos_times):
  pred_med_by_issuance[pd.Timestamp(t)] = oos_pred_med_23[i]
fnf_da_23 = fnf_ds_23["fnf"]

wy_years = list(range(2021, 2026))
obs_cts, obs_sdots, obs_fsum = [], [], []
for yr in wy_years:
  wy_obs = fnf_da_23.sel(time=slice(f"{yr - 1}-10-01", f"{yr}-09-30")).values
  c_o, s_o, f_o = compute_hydrograph_timing_metrics(wy_obs)
  obs_cts.append(c_o)
  obs_sdots.append(s_o)
  obs_fsum.append(f_o)
obs_ct_mean = np.mean(obs_cts, axis=0)
obs_sdot_mean = np.mean(obs_sdots, axis=0)
obs_fsum_mean = np.mean(obs_fsum, axis=0)

print("\n" + "=" * 78)
print(
    "WATER-YEAR HYDROGRAPH TIMING DIAGNOSTICS (WY 2021-2025, 23 SNOW-FED"
    " BASINS)"
)
print("-" * 78)
print(
    f"{'Horizon':<12} | {'Basin CT r':>10} | {'CT MAE (d)':>10} |"
    f" {'SDoT MAE (d)':>12} | {'Jul-Sep f_sum MAE (%)':>21}"
)
print("-" * 78)
for lead in (1, 2, 3):
  l_idx = lead - 1
  pred_cts, pred_sdots, pred_fsum = [], [], []
  for yr in wy_years:
    months_dt = pd.date_range(f"{yr - 1}-10-01", periods=12, freq="MS")
    wy_pred = np.zeros((12, len(b120_to_model_idx)))
    for m_i, target_dt in enumerate(months_dt):
      iss_dt = target_dt - pd.DateOffset(months=l_idx)
      wy_pred[m_i] = pred_med_by_issuance[iss_dt][l_idx, :]
    c_p, s_p, f_p = compute_hydrograph_timing_metrics(wy_pred)
    pred_cts.append(c_p)
    pred_sdots.append(s_p)
    pred_fsum.append(f_p)

  pred_ct_mean = np.mean(pred_cts, axis=0)
  pred_sdot_mean = np.mean(pred_sdots, axis=0)
  pred_fsum_mean = np.mean(pred_fsum, axis=0)
  ct_r = float(np.corrcoef(obs_ct_mean, pred_ct_mean)[0, 1])
  ct_mae = float(np.mean(np.abs(pred_ct_mean - obs_ct_mean)))
  sdot_mae = float(np.mean(np.abs(pred_sdot_mean - obs_sdot_mean)))
  fsum_mae = float(np.mean(np.abs(pred_fsum_mean - obs_fsum_mean)))
  print(
      f"Lead l = {lead:<3d} | {ct_r:>10.3f} | {ct_mae:>10.2f} |"
      f" {sdot_mae:>12.2f} | {fsum_mae:>21.2f}"
  )
print("=" * 78)